In [1]:
import os
import rasterio
import pandas as pd
import numpy as np

# ===================== HELPER FUNCTIONS ===================== #

def find_band_files(root_folder, bands=['B02','B03','B04','B08'], resolution='10m'):
    band_files = {}
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for f in filenames:
            for band in bands:
                if f"{band}_{resolution}.jp2" in f:
                    band_files[band] = os.path.join(dirpath, f)
    return band_files

def find_scl_file(root_folder):
    """Find the SCL (Scene Classification Layer) 20m .jp2 file."""
    for dirpath, _, filenames in os.walk(root_folder):
        for f in filenames:
            if "SCL_20m.jp2" in f:
                return os.path.join(dirpath, f)
    return None

def read_and_normalize_band(band_path):
    with rasterio.open(band_path) as ds:
        band = ds.read(1).astype('float32')
        band_normalized = np.clip(band / 10000.0, 0, 1)
        transform = ds.transform
        shape = band.shape
    return band_normalized, transform, shape

def read_scl_band(scl_path):
    """Read SCL band as integer (no normalization)."""
    with rasterio.open(scl_path) as ds:
        scl = ds.read(1)
        transform = ds.transform
        shape = scl.shape
    return scl, transform, shape

def resample_20m_to_10m(band_20m, transform_20m, shape_20m):
    """
    Resample a 20m band to 10m using nearest-neighbour (2x2 repeat).
    Same method as cell 12.
    """
    rows, cols = shape_20m
    t = transform_20m

    xs_1d = t.c + (np.arange(cols) + 0.5) * t.a
    xs_10m = np.empty(cols * 2, dtype=np.float32)
    xs_10m[0::2] = xs_1d - 5.0
    xs_10m[1::2] = xs_1d + 5.0

    ys_1d = t.f + (np.arange(rows) + 0.5) * t.e
    ys_10m = np.empty(rows * 2, dtype=np.float32)
    ys_10m[0::2] = ys_1d + 5.0
    ys_10m[1::2] = ys_1d - 5.0

    resampled = np.repeat(np.repeat(band_20m, 2, axis=1), 2, axis=0)
    return resampled, xs_10m, ys_10m



def bands_to_dataframe(band_files_10m, band_files_20m, scl_path):
    """
    Combine 10m bands, resampled 20m bands, and resampled SCL into a single DataFrame.
    """
    # --- 10m coordinate grid from first 10m band ---
    first_band = next(iter(band_files_10m.values()))
    band_data, transform, shape = read_and_normalize_band(first_band)
    rows, cols = shape
    row_indices, col_indices = np.indices((rows, cols))
    xs, ys = rasterio.transform.xy(transform, row_indices, col_indices)

    data = {
        "Longitude": np.array(xs).flatten(),
        "Latitude": np.array(ys).flatten()
    }

    # --- 10m bands (B02, B03, B04, B08) ---
    for band_name, path in band_files_10m.items():
        band_data, _, _ = read_and_normalize_band(path)
        data[band_name] = band_data.flatten()

    # --- 20m bands resampled to 10m (B01, B11, B12) ---
    for band_name, path in band_files_20m.items():
        band_20m, transform_20m, shape_20m = read_and_normalize_band(path)
        resampled, _, _ = resample_20m_to_10m(band_20m, transform_20m, shape_20m)
        data[band_name] = resampled.flatten()

    # --- SCL 20m resampled to 10m (nearest neighbour — same 2x2 repeat) ---
    if scl_path:
        scl, scl_transform, scl_shape = read_scl_band(scl_path)
        scl_resampled, _, _ = resample_20m_to_10m(scl, scl_transform, scl_shape)
        scl_flat = scl_resampled.flatten()
        data["ClassID"] = scl_flat
        data["ClassName"] = pd.Series(scl_flat).map(class_dict).values
        print(f"SCL resampled: {scl_shape[0]}x{scl_shape[1]} (20m) → {scl_resampled.shape[0]}x{scl_resampled.shape[1]} (10m)")

    df = pd.DataFrame(data)
    return df

# ===================== SCL CLASS NAMES ===================== #
class_dict = {
    0: "No Data",
    1: "Saturated",
    2: "Dark Area Pixels",
    3: "Cloud Shadow",
    4: "Vegetation",
    5: "Bare Soil",
    6: "Water",
    7: "Unclassified",
    8: "Cloud Medium Probability",
    9: "Cloud High Probability",
    10: "Thin Cirrus",
    11: "Snow / Ice"
}




In [2]:
root_folders=[
            "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2A_MSIL2A_20260311T044241_N0512_R033_T45RYJ_20260311T100110.SAFE_Rangpur/S2A_MSIL2A_20260311T044241_N0512_R033_T45RYJ_20260311T100110.SAFE/GRANULE/L2A_T45RYJ_A055972_20260311T044246/IMG_DATA",
             "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260304T043659_N0512_R033_T45QXG_20260304T083704.SAFE_Meherpur/S2B_MSIL2A_20260304T043659_N0512_R033_T45QXG_20260304T083704.SAFE/GRANULE/L2A_T45QXG_A046963_20260304T044654/IMG_DATA",
             "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260304T043659_N0512_R033_T45QYF_20260304T083704.SAFE_Jessore/S2B_MSIL2A_20260304T043659_N0512_R033_T45QYF_20260304T083704.SAFE/GRANULE/L2A_T45QYF_A046963_20260304T044654/IMG_DATA",
             "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260304T043659_N0512_R033_T45QYG_20260304T083704.SAFE_Pabna/S2B_MSIL2A_20260304T043659_N0512_R033_T45QYG_20260304T083704.SAFE/GRANULE/L2A_T45QYG_A046963_20260304T044654/IMG_DATA",
             "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260304T043659_N0512_R033_T45RXH_20260304T083704.SAFE_Naogoan/S2B_MSIL2A_20260304T043659_N0512_R033_T45RXH_20260304T083704.SAFE/GRANULE/L2A_T45RXH_A046963_20260304T044654/IMG_DATA",
             "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260304T043659_N0512_R033_T45RXJ_20260304T083704.SAFE_Dinajpur/S2B_MSIL2A_20260304T043659_N0512_R033_T45RXJ_20260304T083704.SAFE/S2B_MSIL2A_20260304T043659_N0512_R033_T45RXJ_20260304T083704.SAFE_Dinajpur/GRANULE/L2A_T45RXJ_A046963_20260304T044654/IMG_DATA",
             "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260304T043659_N0512_R033_T45RYH_20260304T083704.SAFE_Bogura/S2B_MSIL2A_20260304T043659_N0512_R033_T45RYH_20260304T083704.SAFE/GRANULE/L2A_T45RYH_A046963_20260304T044654/IMG_DATA",
             "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260311T042659_N0512_R133_T45RZH_20260311T081429.SAFE_Mymensingh/S2B_MSIL2A_20260311T042659_N0512_R133_T45RZH_20260311T081429.SAFE/GRANULE/L2A_T45RZH_A047063_20260311T043543/IMG_DATA",
              "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2C_MSIL2A_20260306T042701_N0512_R133_T46QBM_20260306T071110.SAFE_Dhaka/S2C_MSIL2A_20260306T042701_N0512_R133_T46QBM_20260306T071110.SAFE/GRANULE/L2A_T46QBM_A007825_20260306T044129/IMG_DATA",
              "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2C_MSIL2A_20260306T042701_N0512_R133_T46QCK_20260306T071110.SAFE_Chattogram/S2C_MSIL2A_20260306T042701_N0512_R133_T46QCK_20260306T071110.SAFE/GRANULE/L2A_T46QCK_A007825_20260306T044129/IMG_DATA",
              "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2C_MSIL2A_20260306T042701_N0512_R133_T46QCL_20260306T071110.SAFE_Feni/S2C_MSIL2A_20260306T042701_N0512_R133_T46QCL_20260306T071110.SAFE/GRANULE/L2A_T46QCL_A007825_20260306T044129/IMG_DATA",
              "/kaggle/input/datasets/mezbaussalaheen/dulal-1/S2C_MSIL2A_20260306T042701_N0512_R133_T46RCN_20260306T071110.SAFE_Sylhet/S2C_MSIL2A_20260306T042701_N0512_R133_T46RCN_20260306T071110.SAFE/GRANULE/L2A_T46RCN_A007825_20260306T044129/IMG_DATA"              
              
             ]

In [3]:
import os
import shutil
import json

# Kaggle dataset info
dataset_id = "mezbaussalaheen/filtered-sentinel2-dataset-bangladesh" 
dataset_title = "Filtered Sentinel-2 Dataset"

# Base folder to hold all Parquet files
base_folder = "/kaggle/working/filtered_dataset"
os.makedirs(base_folder, exist_ok=True)

# Create metadata JSON (needed only once)
metadata_path = os.path.join(base_folder, "dataset-metadata.json")
if not os.path.exists(metadata_path):
    metadata = {
        "title": dataset_title,
        "id": dataset_id,
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(metadata_path, "w") as f:
        json.dump(metadata, f)

# Process each root_folder
for idx, root_folder in enumerate(root_folders, start=1):
    print(f"\n[{idx}/{len(root_folders)}] Processing: {root_folder}")

    # idx = idx+5
    
    # --- Process Sentinel-2 data ---
    band_files_10m = find_band_files(root_folder, bands=['B02','B03','B04','B08'], resolution='10m')
    band_files_20m = find_band_files(root_folder, bands=['B01','B11','B12'], resolution='20m')
    scl_path = find_scl_file(root_folder)
    
    df = bands_to_dataframe(band_files_10m, band_files_20m, scl_path)
    
    # Filter ClassID
    filtered_df = df[df['ClassID'].isin([4,5,6])]
    
    # --- Save Parquet inside dataset folder ---
    save_path = os.path.join(base_folder, f"data{idx}.parquet")
    filtered_df.to_parquet(save_path, engine='pyarrow', compression='snappy', index=False)
    print(f"Saved filtered DataFrame to: {save_path} (rows: {filtered_df.shape[0]})")
    
    # Free memory
    del df, filtered_df

print("\nAll batches processed and saved as Parquet.")

# # --- Push to Kaggle dataset once (after all batches are done) ---
# # First upload
# !kaggle datasets create -p {base_folder}


[1/12] Processing: /kaggle/input/datasets/mezbaussalaheen/dulal-1/S2A_MSIL2A_20260311T044241_N0512_R033_T45RYJ_20260311T100110.SAFE_Rangpur/S2A_MSIL2A_20260311T044241_N0512_R033_T45RYJ_20260311T100110.SAFE/GRANULE/L2A_T45RYJ_A055972_20260311T044246/IMG_DATA
SCL resampled: 5490x5490 (20m) → 10980x10980 (10m)
Saved filtered DataFrame to: /kaggle/working/filtered_dataset/data1.parquet (rows: 119162844)

[2/12] Processing: /kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260304T043659_N0512_R033_T45QXG_20260304T083704.SAFE_Meherpur/S2B_MSIL2A_20260304T043659_N0512_R033_T45QXG_20260304T083704.SAFE/GRANULE/L2A_T45QXG_A046963_20260304T044654/IMG_DATA
SCL resampled: 5490x5490 (20m) → 10980x10980 (10m)
Saved filtered DataFrame to: /kaggle/working/filtered_dataset/data2.parquet (rows: 119012792)

[3/12] Processing: /kaggle/input/datasets/mezbaussalaheen/dulal-1/S2B_MSIL2A_20260304T043659_N0512_R033_T45QYF_20260304T083704.SAFE_Jessore/S2B_MSIL2A_20260304T043659_N0512_R033_T45QYF_20260

In [4]:
#2nd phase
!kaggle datasets version -p {base_folder} -m "Added other to the dataset"

Starting upload for file data6.parquet
100%|██████████████████████████████████████| 1.36G/1.36G [00:17<00:00, 82.9MB/s]
Upload successful: data6.parquet (1GB)
Starting upload for file data7.parquet
100%|██████████████████████████████████████| 1.29G/1.29G [00:14<00:00, 93.9MB/s]
Upload successful: data7.parquet (1GB)
Starting upload for file data10.parquet
100%|█████████████████████████████████████████| 812M/812M [00:06<00:00, 130MB/s]
Upload successful: data10.parquet (812MB)
Starting upload for file data9.parquet
100%|██████████████████████████████████████| 1.34G/1.34G [00:16<00:00, 88.2MB/s]
Upload successful: data9.parquet (1GB)
Starting upload for file data3.parquet
100%|████████████████████████████████████████| 716M/716M [00:07<00:00, 99.4MB/s]
Upload successful: data3.parquet (716MB)
Starting upload for file data2.parquet
100%|███████████████████████████████████████| 1.36G/1.36G [00:13<00:00, 109MB/s]
Upload successful: data2.parquet (1GB)
Starting upload for file data1.parquet
1